# NHANES 04 - Intervalos individuales por bootstrap

Aplica la sección 10 de `docs/protocolo_nhanes.md`. El sistema web mostrará, junto a la probabilidad estimada, un intervalo que refleja la **incertidumbre del modelo**: cuánto cambiaría la estimación si el modelo se hubiera entrenado con otra muestra de conglomerados de NHANES.

- 1,000 remuestreos con reemplazo de los 163 conglomerados del desarrollo (`numpy.random.default_rng(42)`).
- En cada uno se reajusta la regresión logística seleccionada **con los hiperparámetros congelados** (notebook 03) y se guardan sus parámetros: valores de imputación, media y desviación del escalado, coeficientes e intercepto.
- Intervalo = percentiles 2.5 y 97.5 de las 1,000 probabilidades.

El intervalo **no** es un rango en el que se encuentre la HbA1c de la persona, ni la probabilidad de un individuo concreto. Es la precisión con que el modelo estima la probabilidad para personas con ese perfil.

**Solo se usa el conjunto de desarrollo.** El modelo congelado no se modifica: la probabilidad central que mostrará la web sigue siendo la del modelo de `models/nhanes/regresion_logistica.joblib`.

In [1]:
from datetime import date
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone

import preparacion_nhanes as pn
from preparacion_nhanes import DESENLACE, PREDICTORES, cargar_desarrollo

pd.set_option('display.float_format', '{:.4f}'.format)
MODELOS = Path('../../models/nhanes')
SALIDA = Path('../../resultados/nhanes/04_bootstrap')
SALIDA.mkdir(parents=True, exist_ok=True)
SEMILLA, REMUESTREOS = 42, 1000

ruta_modelo = MODELOS / 'regresion_logistica.joblib'
artefacto = joblib.load(ruta_modelo)
assert artefacto['seleccionado'] and artefacto['algoritmo'] == 'Regresión logística'
assert artefacto['sha256_desarrollo'] == pn.sha256(pn.RUTA_DESARROLLO), 'los datos de desarrollo cambiaron'

d = cargar_desarrollo()
X, y = d[PREDICTORES], d[DESENLACE].astype(int)
print('Modelo congelado:', ruta_modelo.name, '| hiperparámetros:', artefacto['hiperparametros'], '| AUC CV:', round(artefacto['auc_cv'], 4))

Modelo congelado: regresion_logistica.joblib | hiperparámetros: {'C': 1} | AUC CV: 0.7546


## Parámetros de la regresión logística

El `Pipeline` imputa, escala y aplica la regresión logística. Sus parámetros se pueden guardar como arreglos y la probabilidad se reconstruye con

*logit* = intercepto + Σ coef · (valor imputado − media) / desviación,

lo que permite calcular en la web las 1,000 probabilidades en una sola operación. Primero se comprueba que la reconstrucción reproduce exactamente al modelo congelado.

In [2]:
def parametros(pipeline):
    prep, modelo = pipeline.named_steps['prep'], pipeline.named_steps['modelo']
    filas = {}
    j = 0
    for nombre, sub, columnas in prep.transformers_:
        if nombre == 'remainder':
            continue
        imputar, escalar = sub.named_steps['imputar'], sub.named_steps['escalar']
        for k, col in enumerate(columnas):
            filas[col] = (imputar.statistics_[k], escalar.mean_[k], escalar.scale_[k], modelo.coef_[0, j])
            j += 1
    t = np.array([filas[p] for p in PREDICTORES])
    return {'imputacion': t[:, 0], 'media': t[:, 1], 'escala': t[:, 2], 'coef': t[:, 3],
            'intercepto': float(modelo.intercept_[0])}


def probabilidades(X, imputacion, media, escala, coef, intercepto):
    """X: (n, 8). Parámetros de un modelo (1-D) o de varios remuestreos (2-D) -> (n,) o (n, R)."""
    X = np.asarray(X, dtype=float)[:, None, :] if np.ndim(coef) == 2 else np.asarray(X, dtype=float)
    Xi = np.where(np.isnan(X), imputacion, X)
    logit = ((Xi - media) / escala * coef).sum(axis=-1) + intercepto
    return 1 / (1 + np.exp(-logit))


congelado = parametros(artefacto['modelo'])
diferencia = np.abs(probabilidades(X, **congelado) - artefacto['modelo'].predict_proba(X)[:, 1]).max()
assert diferencia < 1e-10
print(f'Diferencia máxima entre la reconstrucción y el modelo congelado: {diferencia:.1e}')
pd.DataFrame({k: v for k, v in congelado.items() if k != 'intercepto'}, index=PREDICTORES)

Diferencia máxima entre la reconstrucción y el modelo congelado: 3.3e-16


,imputacion,media,escala,coef
edad,42.0000,42.5208,14.0175,0.8010
hombre,0.0000,0.4906,0.4999,0.1292
imc,27.6487,28.7734,6.9004,0.4788
cintura,95.7000,97.2093,15.9723,0.0299
hta,0.0000,0.2415,0.4280,0.0163
familiar_dm,0.0000,0.3787,0.4851,0.1191
actividad,1.0000,0.5210,0.4996,-0.1153
fuma,0.0000,0.2333,0.4229,0.1144


## Bootstrap por conglomerado

In [3]:
conglomerados = d['conglomerado'].unique()
indices = {c: np.flatnonzero(d['conglomerado'].to_numpy() == c) for c in conglomerados}
rng = np.random.default_rng(SEMILLA)

remuestreos = []
for b in range(REMUESTREOS):
    elegidos = rng.choice(conglomerados, size=len(conglomerados), replace=True)
    filas = np.concatenate([indices[c] for c in elegidos])
    ajustado = clone(artefacto['modelo']).fit(X.iloc[filas], y.iloc[filas])
    remuestreos.append(parametros(ajustado))

boot = {k: np.array([r[k] for r in remuestreos]) for k in congelado}
print({k: v.shape for k, v in boot.items()})

{'imputacion': (1000, 8), 'media': (1000, 8), 'escala': (1000, 8), 'coef': (1000, 8), 'intercepto': (1000,)}


In [4]:
ruta_boot = MODELOS / 'bootstrap_regresion_logistica.joblib'
joblib.dump({
    **boot,
    'predictores': PREDICTORES,
    'remuestreos': REMUESTREOS,
    'semilla': SEMILLA,
    'unidad': 'conglomerado (ciclo, SDMVSTRA, SDMVPSU) del conjunto de desarrollo',
    'percentiles': (2.5, 97.5),
    'sha256_modelo': pn.sha256(ruta_modelo),
    'sha256_desarrollo': pn.sha256(pn.RUTA_DESARROLLO),
    'fecha': date.today().isoformat(),
}, ruta_boot, compress=3)
print(ruta_boot.name, f'{ruta_boot.stat().st_size / 1e3:.0f} KB | SHA-256 {pn.sha256(ruta_boot)}')

bootstrap_regresion_logistica.joblib 197 KB | SHA-256 6dc9cb8edc7de680c4c71cbd90cf4daf5e5d35578bfe024b3ba2f38ac62ea25d


### Variabilidad de los coeficientes

Coeficiente estandarizado del modelo congelado y su intervalo percentil bootstrap.

In [5]:
coeficientes = pd.DataFrame({
    'coef_congelado': congelado['coef'],
    'IC95_inf': np.percentile(boot['coef'], 2.5, axis=0),
    'IC95_sup': np.percentile(boot['coef'], 97.5, axis=0),
}, index=PREDICTORES)
coeficientes.to_csv(SALIDA / 'coeficientes_bootstrap.csv')
coeficientes

,coef_congelado,IC95_inf,IC95_sup
edad,0.8010,0.7599,0.8458
hombre,0.1292,0.0946,0.1680
imc,0.4788,0.3987,0.5695
cintura,0.0299,-0.0589,0.1114
hta,0.0163,-0.0159,0.0501
familiar_dm,0.1191,0.0831,0.1581
actividad,-0.1153,-0.1473,-0.0822
fuma,0.1144,0.0749,0.1525


## Intervalos para perfiles de ejemplo

Perfiles ilustrativos, no participantes reales. La probabilidad central es la del modelo congelado; el intervalo proviene de los 1,000 remuestreos.

In [6]:
def perfil(edad, hombre, peso, talla, cintura, hta, familiar_dm, actividad, fuma):
    return {'edad': edad, 'hombre': hombre, 'imc': peso / (talla / 100) ** 2, 'cintura': cintura, 'hta': hta,
            'familiar_dm': familiar_dm, 'actividad': actividad, 'fuma': fuma}


perfiles = pd.DataFrame({
    'Mujer 25 años, 55 kg, 160 cm, cintura 72, activa': perfil(25, 0, 55, 160, 72, 0, 0, 1, 0),
    'Mujer 45 años, 75 kg, 160 cm, cintura 92, familiar con diabetes, activa': perfil(45, 0, 75, 160, 92, 0, 1, 1, 0),
    'Hombre 50 años, 95 kg, 169 cm, cintura 115, HTA, familiar, inactivo, fuma': perfil(50, 1, 95, 169, 115, 1, 1, 0, 1),
    'Hombre 65 años, 100 kg, 170 cm, cintura 118, HTA, familiar, inactivo, fuma': perfil(65, 1, 100, 170, 118, 1, 1, 0, 1),
    'Mismo perfil de 50 años sin antecedente familiar ni dato de cintura': perfil(50, 1, 95, 169, np.nan, 1, 0, 0, 1),
}).T[PREDICTORES]

central = probabilidades(perfiles, **congelado)
todas = probabilidades(perfiles, **boot)
ejemplos = pd.DataFrame({
    'probabilidad': central,
    'IC95_inf': np.percentile(todas, 2.5, axis=1),
    'IC95_sup': np.percentile(todas, 97.5, axis=1),
}, index=perfiles.index)
ejemplos['ancho'] = ejemplos['IC95_sup'] - ejemplos['IC95_inf']
ejemplos.to_csv(SALIDA / 'perfiles_ejemplo.csv')
(100 * ejemplos).round(1)

,probabilidad,IC95_inf,IC95_sup,ancho
"Mujer 25 años, 55 kg, 160 cm, cintura 72, activa",4.0000,3.5000,4.5000,1.0000
"Mujer 45 años, 75 kg, 160 cm, cintura 92, familiar con diabetes, activa",23.0000,21.5000,24.6000,3.1000
"Hombre 50 años, 95 kg, 169 cm, cintura 115, HTA, familiar, inactivo, fuma",54.8000,51.7000,57.6000,5.9000
"Hombre 65 años, 100 kg, 170 cm, cintura 118, HTA, familiar, inactivo, fuma",75.9000,73.2000,78.2000,5.0000
Mismo perfil de 50 años sin antecedente familiar ni dato de cintura,47.7000,44.0000,51.0000,7.1000


## Ancho de los intervalos en el desarrollo

Distribución del ancho del intervalo en los participantes del desarrollo, para saber qué tan precisos serán los intervalos que muestre la web. (Los participantes del desarrollo son los mismos con que se entrenó el modelo; esto describe la precisión del modelo, no su desempeño.)

In [7]:
p_central = probabilidades(X, **congelado)
p_boot = probabilidades(X, **boot)
ancho = np.percentile(p_boot, 97.5, axis=1) - np.percentile(p_boot, 2.5, axis=1)
grupos_p = pd.cut(p_central, [0, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0])
anchos = (pd.DataFrame({'rango_probabilidad': grupos_p, 'ancho_pp': 100 * ancho})
          .groupby('rango_probabilidad', observed=True)['ancho_pp']
          .describe(percentiles=[0.5, 0.9])[['count', 'mean', '50%', '90%', 'max']])
anchos.to_csv(SALIDA / 'ancho_intervalos.csv')
print(f'Ancho del intervalo (puntos porcentuales): mediana {np.median(100 * ancho):.1f}; percentil 90 {np.percentile(100 * ancho, 90):.1f}; máximo {100 * ancho.max():.1f}')
anchos

Ancho del intervalo (puntos porcentuales): mediana 3.7; percentil 90 5.8; máximo 21.2


,count,mean,50%,90%,max
rango_probabilidad,,,,,
"(0.0, 0.1]",3593.0000,1.5337,1.5375,2.0044,3.9255
"(0.1, 0.2]",4493.0000,2.7223,2.6574,3.4367,10.5787
"(0.2, 0.3]",3256.0000,3.9424,3.8587,4.7473,12.6812
"(0.3, 0.4]",2627.0000,4.8872,4.7435,5.8000,14.1209
"(0.4, 0.5]",2034.0000,5.5301,5.3320,6.4372,13.4755
"(0.5, 1.0]",2261.0000,5.9369,5.6850,6.9190,21.2082


## Interpretación

**Reproducción exacta.** Los parámetros guardados reproducen al modelo congelado con una diferencia máxima de 3 × 10⁻¹⁶. La web podrá calcular la probabilidad central y las 1,000 probabilidades bootstrap con los mismos arreglos, sin volver a entrenar nada.

**Coeficientes.** Los intervalos bootstrap de los coeficientes estandarizados de la edad, el sexo, el IMC, el antecedente familiar, la actividad física y el tabaquismo no incluyen el cero. Los del perímetro abdominal y la hipertensión sí lo incluyen, lo que coincide con los OR ajustados del notebook 02.

**Perfiles de ejemplo.** El perfil de 50 años con obesidad, cintura de 115 cm, hipertensión, antecedente familiar, inactividad y tabaquismo obtiene 54.8 % (IC 95 % 51.7–57.6). Es decir, el modelo estima que algo más de la mitad de las personas con ese perfil tendría HbA1c ≥ 5.7 %. Un perfil joven y delgado obtiene 4.0 % (3.5–4.5). Si falta un dato, como la cintura, se imputa con la mediana del desarrollo y el intervalo se ensancha ligeramente.

**Qué indica el ancho.** El intervalo es estrecho: su mediana es de 3.7 puntos porcentuales y se amplía en los perfiles poco frecuentes en NHANES (hasta 21 puntos). Esto se debe a que el modelo se entrenó con 18,264 personas: la probabilidad *promedio* para un perfil está estimada con precisión. Un intervalo estrecho **no** significa que se pueda saber si una persona concreta tiene HbA1c elevada; para eso se requiere la prueba de laboratorio. La web debe expresarlo así.

**Lo que el intervalo no cubre.** Solo refleja la variabilidad por muestreo de NHANES. No incluye la incertidumbre por aplicar el modelo a otra población (Lima), que se trata como limitación de validez externa.

Solo se usó el conjunto de desarrollo; el modelo congelado no se modificó.
